# Plot the output of cellpose_intensities

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition = c("developed" = "#285F62", 
               "failed" = "#CA4F33")

#col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")
col_GFP = "#86AB30"
col_RFP = "#EB5951"

col_unspecified = "#5E5E5E"
col_GATA3 = "#489C9C"
col_NANOG = "#EA9542"
col_neg    = "#8d8d8dff"

col_GFP = "#86AB30"
col_RFP = "#EB5951"

## 1. Extract summary files

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/Human_embryos_aneuploid/output"
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/HE_EXP179/cellpose/output/plots/summarised_results.csv",
"/ceph.groups/mshahbazi.grp/rsakata/HE_EXP186/cellpose/output/plots/summarised_results.csv"
)

merged_df <- analysis_summary_files %>%
  map_dfr(~ read_csv(.x, col_types = cols(sample = col_character())))

In [ ]:
head(merged_df)

In [ ]:
merged_df$sample_name = merged_df$sample

In [ ]:
tbl <- merged_df %>%
  group_by(EXP) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(desc(n_images))  # optional

tbl

In [ ]:
order_cond <- c("developed","failed")

merged_df <- merged_df %>%
  mutate(condition = factor(condition, levels = order_cond))

## Plot 

### A) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 2.5, h = 2.5,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data,aes(x = fct_reorder(!!sample, as.numeric(as.factor(!!condition))), y = !!pct, group = !!condition)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("A_%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# thresholds (edit if you want different cutoffs)

summary_df <- merged_df %>%
  group_by(image, sample_name, condition, karyotype) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3pos, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOGpos, na.rm = TRUE),
    pct_GATA4 = 100 * mean(GATA4pos, na.rm = TRUE),
    pct_negative = 100 * mean(!(GATA3pos | NANOGpos | GATA4pos), na.rm = TRUE),
    pct_double_GATA3_NANOG = 100 * mean(GATA3pos & NANOGpos, na.rm = TRUE),
    .groups = "drop"
  )

head(summary_df)


In [ ]:
order_cond <- c("developed", "failed")

summary_df <- summary_df %>%
  mutate(condition = factor(condition, levels = order_cond))

In [ ]:
# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "pctNANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, palette = col_condition,out_dir = out_dir,
                    title = "pctGATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "pctnegative")




In [ ]:
# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, sample = karyotype, palette = col_condition, out_dir = out_dir, 
                    title = "pctGATA3+_karyotype")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, sample = karyotype, palette = col_condition,out_dir = out_dir,
                    title = "pctNANOG+_karyotype")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, sample = karyotype, palette = col_condition,out_dir = out_dir,
                    title = "pctGATA4+_karyotype")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, sample = karyotype, palette = col_condition,out_dir = out_dir, 
                    title = "pctnegative_karyotype")

### B) Average intensity per condition

In [ ]:
plot_pct_bar_points <- function(
  data,                               
  pct = pct_GATA3,                    
  condition = condition,              
  out_dir,                    
  title = NULL,                       
  palette = "black",                     
  w = 1, h = 2,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  error_bar_width = 0.2,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.1
) {
  pct       <- enquo(pct)
  condition <- enquo(condition)

  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # 1. Calculate Summary Stats by CONDITION (Mean & SD)
  cond_summary <- data %>%
    group_by(!!condition) %>%
    summarise(
      mean_val = mean(!!pct, na.rm = TRUE),
      sd_val   = sd(!!pct, na.rm = TRUE),
      .groups  = "drop"
    )

  # 2. Build Plot
  # We map X to the Condition. 
  # Note: ensure your 'condition' column levels are set correctly before running this if you want specific order.
  p <- ggplot(data, aes(x = !!condition, y = !!pct)) +
    
    # A. The Bar (Mean of the condition)
    geom_col(
      data = cond_summary,
      aes(y = mean_val, fill = !!condition),
      width = bar_width,
      alpha = 0.6,           # Slight transparency to see points better
      show.legend = FALSE,
      fill = "grey"
    ) +
    
    # B. The Error Bars (Mean +/- SD)
    geom_errorbar(
      data = cond_summary,
      aes(
        y = mean_val, 
        ymin = pmax(0, mean_val - sd_val), # pmax(0, ...) prevents error bar going below 0
        ymax = mean_val + sd_val
      ),
      width = error_bar_width
    ) +
    
    # C. The Individual Points (Jittered)
    geom_jitter(
      size = point_size, 
      alpha = point_alpha,
      width = jitter_width,
      height = 0,             # Don't jitter vertically (keeps Y value accurate)
      na.rm = TRUE,
      color =  palette
    ) +
    
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
      #scale_fill_manual(values=col_condition)+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("B_%s_by_condition.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# Define order first (optional)
summary_df$condition <- factor(summary_df$condition, levels = c("developed", "failed"))

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_GATA3, out_dir = out_dir, 
                    title = "pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_NANOG,out_dir = out_dir,
                    title = "pctNANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, palette = "purple",out_dir = out_dir,
                    title = "pctGATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_unspecified,out_dir = out_dir, 
                    title = "pctnegative")

# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, palette = "black",out_dir = out_dir, 
                    title = "pct_GATA3+_NANOG+")


### Plot by ploidy

In [ ]:
summary_df<- summary_df %>%
  mutate(
    n_chr_affected = str_count(karyotype, "[+-]"),
    ploidy = case_when(
      n_chr_affected > 1 ~ "complex",
      str_detect(karyotype, "\\+") ~ "trisomy",
      str_detect(karyotype, "-") ~ "monosomy",
      TRUE ~ NA_character_
    )
  )
summary_df

In [ ]:
# Define order first (optional)
summary_df$ploidy <- factor(summary_df$ploidy, levels = c("trisomy", "monosomy", "complex"))

# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, condition = ploidy, palette = col_GATA3, out_dir = out_dir, 
                    title = "pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, condition = ploidy, palette = col_NANOG,out_dir = out_dir,
                    title = "pctNANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, condition = ploidy, palette = "purple",out_dir = out_dir,
                    title = "pctGATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, condition = ploidy, palette = col_unspecified,out_dir = out_dir, 
                    title = "pctnegative")





In [ ]:
check_test <- function(
  data,
  group_var = "ploidy",
  value_var = "pct_negative",
  conditions = c("trisomy", "monosomy", "complex"),
  alpha = 0.05
) {

  g <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    split(.[[group_var]]) %>%
    lapply(function(d) na.omit(d[[value_var]]))

  g <- g[conditions]

  n1 <- length(g[[1]])
  n2 <- length(g[[2]])

  if (n1 < 3 || n2 < 3) {
    return(tibble(
      group1 = conditions[1],
      group2 = conditions[2],
      n1 = n1,
      n2 = n2,
      shapiro_p1 = NA_real_,
      shapiro_p2 = NA_real_,
      variance_p = NA_real_,
      normal = NA,
      recommended = "Too few points to assess normality"
    ))
  }

  sp1 <- tryCatch(shapiro.test(g[[1]])$p.value,
                  error = function(e) NA_real_)
  sp2 <- tryCatch(shapiro.test(g[[2]])$p.value,
                  error = function(e) NA_real_)

  normal <- all(!is.na(c(sp1, sp2))) &&
    sp1 > alpha &&
    sp2 > alpha

  variance_p <- tryCatch(
    var.test(g[[1]], g[[2]])$p.value,
    error = function(e) NA_real_
  )

  tibble(
    group1 = conditions[1],
    group2 = conditions[2],
    n1 = n1,
    n2 = n2,
    shapiro_p1 = sp1,
    shapiro_p2 = sp2,
    variance_p = variance_p,
    normal = normal,
    recommended = ifelse(
      normal,
      "Welch t-test",
      "Wilcoxon rank-sum test"
    )
  )
}

check_test(summary_df)

In [ ]:
wilcox_res <- summary_df %>%
 # filter(sample_name %in% c("Developed", "Poor Quality")) %>%
  #mutate(sample_name = droplevels(sample_name)) %>%
  #group_by(state) %>%
  wilcox_test(
    pct_negative ~ ploidy
    #ref.group = "G_R",
    #p.adjust.method = "holm"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
kw_res <- summary_df %>%
 #group_by(state) %>%
  kruskal_test(pct_negative ~ ploidy) %>%
  mutate(
    stars = case_when(
      p < 0.0001 ~ "****",
      p < 0.001  ~ "***",
      p < 0.01   ~ "**",
      p < 0.05   ~ "*",
      TRUE       ~ "ns"
    )
  )
kw_res

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    pct_GATA3 ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    pct_NANOG ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    pct_GATA4 ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

### C) Normalised Intensities

In [ ]:
title   <- "GATA3_norm_intensity"
w <- 1.5; h <- 3.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

# Define order first (optional)
merged_df$condition <- factor(merged_df$condition, levels = c( "Poor Quality", "Developed"))


p <- ggplot(merged_df %>% filter(GATA3pos == TRUE), aes(x = forcats::fct_reorder(sample_name, as.numeric(condition)), y = GATA3_norm_ctr, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.1, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")

ggsave(file.path(out_dir, sprintf("C_%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "NANOG_norm_intensity"
w <- 1.5; h <- 3.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA3pos == FALSE), aes(x = forcats::fct_reorder(sample_name, as.numeric(condition)), y = NANOG_norm_ctr, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.1, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")


ggsave(file.path(out_dir, sprintf("C_%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "GATA4_norm_intensity"
w <- 1.5; h <- 3.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA3pos == FALSE), aes(x = forcats::fct_reorder(sample_name, as.numeric(condition)), y = GATA4_norm_ctr, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.1, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA4 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")


ggsave(file.path(out_dir, sprintf("C_%s.pdf", title)), plot = p, width = w, height = h)
p

### D) Plot intesity by condition

In [ ]:
# thresholds (edit if you want different cutoffs)
summary_df <- merged_df %>%
  group_by(image, sample_name, condition) %>%
  summarise(
    n = n(),
    GATA3 = mean(GATA3_norm_ctr, na.rm = TRUE),
    NANOG = mean(NANOG_norm_ctr, na.rm = TRUE),
    GATA4 = mean(GATA4_norm_ctr, na.rm = TRUE),
    .groups = "drop"
  )

head(summary_df)

In [ ]:
summary_df$condition <- factor(summary_df$condition, levels = c(  "Developed", "Poor Quality"))

In [ ]:
title   <- "GATA3"
w <- 1; h <- 2
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(summary_df, aes(x = condition, y = GATA3, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # pointsa with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 


ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p

In [ ]:
title   <- "NANOG"
w <- 1; h <- 2
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(summary_df, aes(x = condition, y = NANOG, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # pointsa with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 


ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "GATA4"
w <- 1; h <- 2
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(summary_df, aes(x = condition, y = GATA4, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # pointsa with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA4 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  #coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      legend.position = "none"
    ) 


ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p

In [ ]:
check_test <- function(
  data,
  group_var = "condition",
  value_var = "NANOG",
  conditions = c("Developed", "Poor Quality"),
  alpha = 0.05
) {

  g <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    split(.[[group_var]]) %>%
    lapply(function(d) na.omit(d[[value_var]]))

  g <- g[conditions]

  n1 <- length(g[[1]])
  n2 <- length(g[[2]])

  if (n1 < 3 || n2 < 3) {
    return(tibble(
      group1 = conditions[1],
      group2 = conditions[2],
      n1 = n1,
      n2 = n2,
      shapiro_p1 = NA_real_,
      shapiro_p2 = NA_real_,
      variance_p = NA_real_,
      normal = NA,
      recommended = "Too few points to assess normality"
    ))
  }

  sp1 <- tryCatch(shapiro.test(g[[1]])$p.value,
                  error = function(e) NA_real_)
  sp2 <- tryCatch(shapiro.test(g[[2]])$p.value,
                  error = function(e) NA_real_)

  normal <- all(!is.na(c(sp1, sp2))) &&
    sp1 > alpha &&
    sp2 > alpha

  variance_p <- tryCatch(
    var.test(g[[1]], g[[2]])$p.value,
    error = function(e) NA_real_
  )

  tibble(
    group1 = conditions[1],
    group2 = conditions[2],
    n1 = n1,
    n2 = n2,
    shapiro_p1 = sp1,
    shapiro_p2 = sp2,
    variance_p = variance_p,
    normal = normal,
    recommended = ifelse(
      normal,
      "Welch t-test",
      "Wilcoxon rank-sum test"
    )
  )
}

check_test(summary_df)

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    GATA3 ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    GATA4 ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

In [ ]:
wilcox_res <- summary_df %>%
  wilcox_test(
    NANOG ~ condition
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

wilcox_res

## Save 

In [ ]:

write_csv(merged_df, file.path(out_dir, "summarised_results.csv"))